# 証券営業インテリジェンス ハンズオン
## Part 1: Cortex AI セキュリティ・アクセス制御

このパートでは、金融機関において特に重要な **Snowflake Cortex AI のセキュリティ設定**を実践します。

### ハンズオン全体構成

| # | ノートブック | 内容 | 目安時間 |
|---|---|---|---|
| 2 | **Part 1（本書）** | Cortex AI セキュリティ・アクセス制御 | 20分 |
| 3 | Part 2 | Cortex AI Functions（要約・感情分析・分類） | 60分 |
| 4 | Part 3 | ファンド目論見書解析（AI_PARSE_DOCUMENT） | 30分 |
| 5 | Part 4 | Cortex Analyst（自然言語 to SQL） | 45分 |
| 6 | Part 5 | Cortex Search（セマンティック検索） | 45分 |
| 7 | Part 6 | Cortex Agent（Snowflake Intelligence） | 45分 |

### 前提条件
- `setup.sql` 実行済み（環境セットアップ・データ投入完了）

### 3層のアクセス制御

```
[Layer 1] CORTEX_ENABLED_CROSS_REGION  ─── データを処理するリージョンを制限
[Layer 2] CORTEX_MODELS_ALLOWLIST      ─── 使用可能なモデルをホワイトリストで管理
[Layer 3] Model RBAC (ロール別モデル制御)  ─── ロールごとに使用可能モデルを制御
                    +
[Layer 4] SNOWFLAKE.CORTEX_USER        ─── AI Functions 実行権限の付与・剥奪
[Layer 5] Masking Policy + AI_REDACT   ─── LLMに送るデータのPIIマスキング
```

### 🏦 金融機関（AWS利用）における推奨構成

> - AWSを利用している場合、Claude（Anthropic）モデルは **AWS US リージョン** で稼働
> - 日本リージョンのアカウントから Claude を使うには `CORTEX_ENABLED_CROSS_REGION = 'AWS_US'` が必要
> - 未承認モデルの使用を防ぐために `CORTEX_MODELS_ALLOWLIST` でホワイトリストを設定

> ⚠️ このノートブックの大部分は `ACCOUNTADMIN` 権限が必要です。

> ⏱️ **このパートの目安時間: 20分**

In [ ]:
%%sql -r result_env
USE ROLE ACCOUNTADMIN;
USE DATABASE SNOWFINANCE_DB;
USE SCHEMA DEMO_SCHEMA;
USE WAREHOUSE DEMO_WH;
SELECT CURRENT_ROLE() AS "ロール", CURRENT_REGION() AS "リージョン",
       CURRENT_DATABASE() AS DB, CURRENT_WAREHOUSE() AS WH;

## Layer 1: クロスリージョン推論の設定

### CORTEX_ENABLED_CROSS_REGION パラメータ

| 設定値 | 意味 | Claudeが使えるか？ |
|---|---|---|
| `DISABLED` | **デフォルト**。同一リージョンのモデルのみ | ❌（通常、日本リージョンにはない） |
| `ANY_REGION` | 全リージョンを許可 | ✅ |
| `AWS_US` | AWS 米国リージョンを追加許可 | ✅（Claudeは AWS Bedrock経由でUS稼働） |
| `AWS_EU` | AWS EUリージョンを追加許可 | 一部のみ |
| `AWS_US,AWS_EU` | 複数リージョンをカンマ区切りで指定 | ✅ |

> 💡 **AWS利用金融機関向けの推奨設定**: `CORTEX_ENABLED_CROSS_REGION = 'AWS_US'`
> データはAWS内ネットワーク（暗号化済み）のみを経由し、公衆インターネットを通りません。

> ⏱️ **目安: 4分**

In [ ]:
%%sql -r result_cross_region
-- 現在のクロスリージョン設定を確認
SHOW PARAMETERS LIKE 'CORTEX_ENABLED_CROSS_REGION' IN ACCOUNT;

In [ ]:
%%sql -r result_set_cross_region
-- ハンズオン用: ANY_REGION に設定（全モデルを使用可能にする）
-- 本番環境では 'AWS_US' または 'DISABLED' を推奨
ALTER ACCOUNT SET CORTEX_ENABLED_CROSS_REGION = 'ANY_REGION';

-- 設定確認
SHOW PARAMETERS LIKE 'CORTEX_ENABLED_CROSS_REGION' IN ACCOUNT;

## Layer 2: モデルホワイトリスト (CORTEX_MODELS_ALLOWLIST)

アカウント全体で**使用可能なモデルをホワイトリストで制限**できます。
未承認モデルの使用をシステム的に防ぎ、コスト管理とガバナンスを強化します。

```sql
-- 全モデルを許可（デフォルト）
ALTER ACCOUNT SET CORTEX_MODELS_ALLOWLIST = 'All';

-- 特定モデルのみ許可（金融機関向け推奨）
ALTER ACCOUNT SET CORTEX_MODELS_ALLOWLIST = 'claude-sonnet-4-6,claude-opus-4-6';

-- 全モデルを禁止（Model RBAC のみで制御する場合）
ALTER ACCOUNT SET CORTEX_MODELS_ALLOWLIST = 'None';
```

> ⏱️ **目安: 4分**

In [ ]:
%%sql -r result_allowlist_check
-- 現在のモデルホワイトリスト設定を確認
SHOW PARAMETERS LIKE 'CORTEX_MODELS_ALLOWLIST' IN ACCOUNT;

In [ ]:
%%sql -r result_set_allowlist
-- ハンズオン用: Claudeモデルのみをホワイトリストに設定
ALTER ACCOUNT SET CORTEX_MODELS_ALLOWLIST = 'claude-sonnet-4-6,claude-opus-4-6';

-- 設定確認
SHOW PARAMETERS LIKE 'CORTEX_MODELS_ALLOWLIST' IN ACCOUNT;

In [ ]:
-- ホワイトリスト外 & Model RBAC 未許可モデルを試みる（エラーになることを確認）
-- ※ ACCOUNTADMIN はホワイトリスト制限をバイパスできるため、制限付きロールで検証する
-- ※ セカンダリーロールも無効化しないと他ロールの権限が引き継がれる
USE ROLE DEMO_CLAUDE_ONLY_ROLE;
USE SECONDARY ROLES NONE;

SELECT AI_COMPLETE('openai-gpt-5.1', '日本経済について一言') AS "応答";
-- → Error: Model does not exist or is not authorized.

In [ ]:
-- ホワイトリスト内モデルは正常動作
SELECT AI_COMPLETE('claude-sonnet-4-6', 'Snowflakeを50文字で表現して') AS "応答";

In [ ]:
-- ハンズオン用: ホワイトリストを元に戻す（全モデル許可）
USE ROLE ACCOUNTADMIN;
ALTER ACCOUNT SET CORTEX_MODELS_ALLOWLIST = 'All';
SHOW PARAMETERS LIKE 'CORTEX_MODELS_ALLOWLIST' IN ACCOUNT;

## Layer 3: ロール別モデルアクセス制御（Model RBAC）

`CORTEX_MODELS_ALLOWLIST` がアカウント全体の制御なのに対し、
**Model RBAC** はロールごとに細かく使用可能モデルを制御できます。

### 設定フロー

```
1. CORTEX_BASE_MODELS_REFRESH()  → SNOWFLAKE.MODELS スキーマにモデルオブジェクトを生成
2. SHOW MODELS IN SNOWFLAKE.MODELS  → 利用可能モデルと対応アプリケーションロールを確認
3. GRANT APPLICATION ROLE SNOWFLAKE."CORTEX-MODEL-ROLE-<MODEL>"
       TO ROLE <user_role>         → ロールに特定モデルのアクセスを付与
```

> 💡 **CORTEX_MODELS_ALLOWLIST との使い分け**:
> - アカウント全体で禁止 → `CORTEX_MODELS_ALLOWLIST = 'None'` にして Model RBAC のみで制御
> - ロールAは小モデルのみ、ロールBは大モデルも使える → Model RBAC

> ⏱️ **目安: 5分**

In [ ]:
%%sql -r result_model_refresh
-- Step 1: SNOWFLAKE.MODELS スキーマにモデルオブジェクトを生成
-- ※ 新しいモデルが追加されたときも再実行で更新できる
CALL SNOWFLAKE.MODELS.CORTEX_BASE_MODELS_REFRESH();

In [ ]:
%%sql -r result_show_models
-- Step 2: 利用可能なモデルオブジェクトを確認
SHOW MODELS IN SNOWFLAKE.MODELS;

In [ ]:
%%sql -r result_app_roles
-- Claudeモデルに対応したアプリケーションロールを確認
SHOW APPLICATION ROLES IN APPLICATION SNOWFLAKE;

In [ ]:
%%sql -r result_grant_claude_role
-- Step 3a: 特定ロールに claude-sonnet-4-6 のみを許可する例
-- （デモ用ロール作成 → Claudeモデルロールのみ付与）

CREATE ROLE IF NOT EXISTS DEMO_CLAUDE_ONLY_ROLE;
GRANT USAGE ON DATABASE SNOWFINANCE_DB TO ROLE DEMO_CLAUDE_ONLY_ROLE;
GRANT USAGE ON SCHEMA DEMO_SCHEMA TO ROLE DEMO_CLAUDE_ONLY_ROLE;
GRANT USAGE ON WAREHOUSE DEMO_WH TO ROLE DEMO_CLAUDE_ONLY_ROLE;

-- Cortex AI Functions 実行権限を付与
GRANT DATABASE ROLE SNOWFLAKE.CORTEX_USER TO ROLE DEMO_CLAUDE_ONLY_ROLE;

-- Claudeモデルのみ許可（Model RBAC）
GRANT APPLICATION ROLE SNOWFLAKE."CORTEX-MODEL-ROLE-CLAUDE-SONNET-4-6"
    TO ROLE DEMO_CLAUDE_ONLY_ROLE;

SELECT 'DEMO_CLAUDE_ONLY_ROLE: claude-sonnet-4-6 のみ使用可能' AS STATUS;

In [ ]:
-- Model RBAC の動作確認: DEMO_CLAUDE_ONLY_ROLE に切り替えてテスト
-- まず現在のユーザーにロールを付与
SET current_user_name = CURRENT_USER();
GRANT ROLE DEMO_CLAUDE_ONLY_ROLE TO USER IDENTIFIER($current_user_name);
USE ROLE DEMO_CLAUDE_ONLY_ROLE;
USE SECONDARY ROLES NONE;

-- ✅ 許可済みモデル（claude-sonnet-4-6）→ 成功するはず
SELECT AI_COMPLETE(
    'claude-sonnet-4-6',
    'Snowflakeを一言で説明して'
) AS "許可モデル_応答";

In [ ]:
-- Step 3b: 全モデルを使えるロールへの付与
USE ROLE ACCOUNTADMIN;
-- GRANT APPLICATION ROLE SNOWFLAKE."CORTEX-MODEL-ROLE-ALL" TO ROLE ANALYST_FULL_ROLE;

-- 現在のロールに付与されているモデルロールを確認
SHOW GRANTS TO ROLE DEMO_CLAUDE_ONLY_ROLE;

## Layer 4: SNOWFLAKE.CORTEX_USER（AI Functions 実行権限）

`SNOWFLAKE.CORTEX_USER` データベースロールを持つロールのみが
`AI_COMPLETE`, `AI_CLASSIFY` などの AI Functions を実行できます。
Model RBAC と組み合わせることで、**「使える機能」と「使えるモデル」を独立して制御**できます。

> ⏱️ **目安: 2分**

In [ ]:
%%sql -r result_cortex_user_grants
-- CORTEX_USER を付与されているロールを確認
SHOW GRANTS OF DATABASE ROLE SNOWFLAKE.CORTEX_USER;

In [ ]:
%%sql -r result_ai_audit
-- ロールへのCortex権限付与（参考）
-- AI Functions を許可:
-- GRANT DATABASE ROLE SNOWFLAKE.CORTEX_USER TO ROLE <role_name>;
-- AI Functions を禁止（取り消し）:
-- REVOKE DATABASE ROLE SNOWFLAKE.CORTEX_USER FROM ROLE <role_name>;

-- AI 使用状況の監査（直近セッション）
SELECT
    START_TIME AS "実行日時",
    USER_NAME AS "ユーザー",
    ROLE_NAME AS "ロール",
    LEFT(QUERY_TEXT, 120) AS "クエリ先頭",
    EXECUTION_STATUS AS "ステータス"
FROM TABLE(INFORMATION_SCHEMA.QUERY_HISTORY_BY_SESSION(RESULT_LIMIT => 20))
WHERE (UPPER(QUERY_TEXT) LIKE '%SNOWFLAKE.CORTEX%'
    OR UPPER(QUERY_TEXT) LIKE '%AI_COMPLETE%'
    OR UPPER(QUERY_TEXT) LIKE '%AI_CLASSIFY%'
    OR UPPER(QUERY_TEXT) LIKE '%AI_SENTIMENT%')
  AND EXECUTION_STATUS = 'SUCCESS'
ORDER BY START_TIME DESC;

## まとめ

Part 1 完了！4層のセキュリティ設定を実践しました。

### セキュリティ設定サマリー

| Layer | パラメータ / 機能 | 今回の設定 | 本番推奨 |
|---|---|---|---|
| 1 | `CORTEX_ENABLED_CROSS_REGION` | `ANY_REGION`（ハンズオン用） | `AWS_US`（Claudeを使うなら必須） |
| 2 | `CORTEX_MODELS_ALLOWLIST` | `All`（元に戻した） | `claude-sonnet-4-6,claude-opus-4-6` など |
| 3 | Model RBAC | `CORTEX-MODEL-ROLE-CLAUDE-SONNET-4-6` を付与 | ロール設計に合わせて付与 |
| 4 | `SNOWFLAKE.CORTEX_USER` | AI Functions 実行権限 | 業務ロールのみに限定 |

### 設定コマンドリファレンス

```sql
-- Layer 1: クロスリージョン
ALTER ACCOUNT SET CORTEX_ENABLED_CROSS_REGION = 'AWS_US';  -- AWSのみ利用金融機関向け

-- Layer 2: モデルホワイトリスト
ALTER ACCOUNT SET CORTEX_MODELS_ALLOWLIST = 'claude-sonnet-4-6,claude-opus-4-6';

-- Layer 3: Model RBAC
CALL SNOWFLAKE.MODELS.CORTEX_BASE_MODELS_REFRESH();
GRANT APPLICATION ROLE SNOWFLAKE."CORTEX-MODEL-ROLE-CLAUDE-SONNET-4-6" TO ROLE MY_ROLE;

-- Layer 4: Cortex実行権限
GRANT DATABASE ROLE SNOWFLAKE.CORTEX_USER TO ROLE MY_ROLE;
```

**次のステップ:** `part2_ai_functions.ipynb` で Cortex AI Functions の実践的な活用を体験してください。

In [ ]:
%%sql -r result_restore_role
-- テスト後は ACCOUNTADMIN に戻す
USE ROLE ACCOUNTADMIN;
SELECT 'ACCOUNTADMIN に戻しました' AS STATUS;